# Verificación de la capa Bronze — Iceberg

Revisamos las tablas Iceberg en `nessie.bronze` tras la ingesta:
referencias Nessie, metadatos, snapshots, particionado y calidad de datos.

In [ ]:
from common.spark_factory import get_spark
from common import nessie_manager as nessie

spark = get_spark("verify-bronze-iceberg")
spark.version

## 1. Catálogo Nessie — Referencias y namespaces

La ingesta bronze usa WAP: cada mes crea una rama efímera, escribe, valida y
hace merge a `bronze`. Tras el último mes de cada año se crea un tag de auditoría.

In [ ]:
# Referencias Nessie (ramas + tags)
print("Nessie references:\n")
nessie.print_references(spark)

refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"
branches = refs_df.filter(refs_df[type_col].ilike("branch")).count()
tags = refs_df.filter(refs_df[type_col].ilike("tag")).count()
print(f"\n  Branches: {branches}  |  Tags: {tags}")

In [ ]:
# Switch to bronze branch for the rest of the notebook
nessie.set_spark_branch(spark, "bronze")

# Available namespaces
spark.sql("SHOW NAMESPACES IN nessie").show(truncate=False)

In [ ]:
# Tables in the bronze namespace
spark.sql("SHOW TABLES IN nessie.bronze").show(truncate=False)

## 2. Yellow Taxi — Propiedades y esquema

In [ ]:
# Esquema, particionado, propiedades
spark.sql("DESCRIBE EXTENDED nessie.bronze.yellow_taxi").show(250, truncate=False)

In [ ]:
# Propiedades Iceberg (format version, gc, etc.)
spark.sql("SHOW TBLPROPERTIES nessie.bronze.yellow_taxi").show(truncate=False)

## 3. Snapshots e historial

In [ ]:
# Cada write/overwrite crea un snapshot
spark.sql("SELECT * FROM nessie.bronze.yellow_taxi.snapshots").show(truncate=False)

In [ ]:
# History (time-ordered snapshot log)
spark.sql("SELECT * FROM nessie.bronze.yellow_taxi.history").show(truncate=False)

## 3b. Crecimiento de registros por tag

Cada tag Nessie captura el estado de `bronze` tras ingestar todos los meses
de un año. Leyendo la tabla en cada tag vemos cómo crece año a año.

In [ ]:
# Conteo de registros en cada tag anual para ver el crecimiento incremental
import re

refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"

tag_rows_df = refs_df.filter(refs_df[type_col].ilike("tag")) \
    .filter(refs_df["name"].rlike(r"^bronze-yellow_taxi-\d{4}$"))
tag_names = [r["name"] for r in tag_rows_df.collect()]

tag_pattern = re.compile(r"bronze-yellow_taxi-(\d{4})")
tag_rows = []
for name in tag_names:
    m = tag_pattern.match(name)
    if m:
        tag_rows.append((name, int(m.group(1))))
tag_rows.sort(key=lambda t: t[1])

growth = []
for tag_name, year in tag_rows:
    nessie.set_spark_branch(spark, tag_name)
    count = spark.table("nessie.bronze.yellow_taxi").count()
    growth.append((tag_name, year, count))

nessie.set_spark_branch(spark, "bronze")

growth_df = spark.createDataFrame(
    [(tag, year, total, total - (growth[i-1][2] if i > 0 else 0))
     for i, (tag, year, total) in enumerate(growth)],
    ["tag", "year", "cumulative_records", "records_added"]
)
growth_df.show(truncate=False)

## 4. Particionado y ficheros

In [ ]:
# Partition spec
spark.sql("SELECT * FROM nessie.bronze.yellow_taxi.partitions").show(truncate=False)

In [ ]:
# Ficheros de datos: rutas, tamaños, registros por fichero
files_df = spark.sql("SELECT * FROM nessie.bronze.yellow_taxi.files")
files_df.select(
    "content", "file_path", "file_format",
    "record_count", "file_size_in_bytes"
).show(100, truncate=80)

In [ ]:
# Resumen: ficheros totales, tamaño, registros
from pyspark.sql.functions import count, sum as spark_sum, round as spark_round

files_df.select(
    count("*").alias("total_files"),
    spark_round(spark_sum("file_size_in_bytes") / 1024 / 1024, 2).alias("total_size_mb"),
    spark_sum("record_count").alias("total_records"),
).show()

## 5. Manifiestos

In [ ]:
# Una entrada por snapshot
spark.sql("SELECT * FROM nessie.bronze.yellow_taxi.manifests").show(truncate=80)

## 6. Calidad de datos

In [ ]:
df = spark.table("nessie.bronze.yellow_taxi")
print(f"Total records: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.printSchema()

In [ ]:
# Registros por año-mes (valida el particionado)
from pyspark.sql.functions import year, month

df.groupBy(
    year("tpep_pickup_datetime").alias("year"),
    month("tpep_pickup_datetime").alias("month")
).count() \
 .orderBy("year", "month") \
 .show(50)

In [ ]:
# Nulos por columna
from pyspark.sql.functions import col as c, sum as spark_sum

null_counts = df.select([
    spark_sum(c(col).isNull().cast("int")).alias(col)
    for col in df.columns
])
null_counts.show(vertical=True)

In [ ]:
# Estadísticas básicas
df.select(
    "trip_distance", "fare_amount", "tip_amount",
    "total_amount", "passenger_count"
).summary("count", "min", "max", "mean").show()

In [ ]:
# Sample rows
df.show(5, truncate=False)

## 7. Taxi Zone Lookup

In [ ]:
# Check if the reference table exists and inspect it
try:
    zones = spark.table("nessie.bronze.taxi_zone_lookup")
    print(f"Records: {zones.count()}")
    zones.printSchema()
    zones.show(10, truncate=False)
except Exception as e:
    print(f"Table not yet ingested: {e}")

In [ ]:
spark.stop()